# Phase 1: NIFTY-50 Exploratory Data Analysis (EDA)
This notebook performs the exploratory data analysis on the NIFTY-50 historical equity prices and benchmarks. 
It addresses all Phase 1 dataset requirements:
1. **Dataset Overview**: Detailed look at files, structures, and indices.
2. **Missing Value Analysis**: Locating missing data points and Trades/Volume sparsity.
3. **Stock Count Analysis**: Active stock counts over time.
4. **Sector Distribution Analysis**: Weights of different industries represented in the index.
5. **Price & Returns Distribution Analysis**: Log return distributions, skewness, and kurtosis.
6. **Correlation Analysis**: Stock returns co-movements.
7. **Market Regime Analysis**: Index trends vs India VIX.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Append parent directory to sys.path so 'src' is importable
sys.path.append("..")

from src.config import SYMBOLS, SECTOR_MAP
from src.data_loader import load_metadata, load_stock_data, load_index_data, create_aligned_panel

# Set default plot styles
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "ggplot")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True

## 1. Dataset Overview
We load stock metadata, the NIFTY 50 index benchmark, and the India VIX volatility index to understand the files and their structural summaries.

In [ ]:
# Load constituent sector metadata
metadata_df = load_metadata()
print("=== STOCK METADATA SUMMARY ===")
print(f"Shape: {metadata_df.shape}")
print(metadata_df.head())

# Load NIFTY 50 index benchmark
nifty50_df = load_index_data("NIFTY 50")
print("\n=== NIFTY 50 INDEX BENCHMARK SUMMARY ===")
print(f"Shape: {nifty50_df.shape}")
print(nifty50_df.head())

# Load India VIX volatility index
vix_df = load_index_data("INDIA VIX")
print("\n=== INDIA VIX VOLATILITY BENCHMARK SUMMARY ===")
print(f"Shape: {vix_df.shape}")
print(vix_df.head())

# Load a sample stock (INFY) to inspect columns and data types
infy_df = load_stock_data("INFY")
print("\n=== SAMPLE STOCK (INFY) RECORDS ===")
print(f"Shape: {infy_df.shape}")
print(infy_df.dtypes)

## 2. Missing Value Analysis
We analyze missing data across all stock files, checking which columns contain NaNs and looking at early-stage Trades and volume sparsity.

In [ ]:
missing_percentages = []
for symbol in SYMBOLS:
    stock_df = load_stock_data(symbol)
    if not stock_df.empty:
        null_counts = stock_df.isnull().sum()
        null_pct = (null_counts / len(stock_df)) * 100
        row_dict = null_pct.to_dict()
        row_dict['Symbol'] = symbol
        row_dict['Total_Rows'] = len(stock_df)
        missing_percentages.append(row_dict)

missing_df = pd.DataFrame(missing_percentages).set_index('Symbol')
mean_missing = missing_df.mean().drop('Total_Rows')
print("=== AVERAGE MISSING % PER COLUMN ACROSS ALL NIFTY-50 STOCKS ===")
print(mean_missing)

# Display stocks with the highest volume of missing Trades data
print("\n=== TOP STOCKS WITH MISSING 'TRADES' RECORDS ===")
print(missing_df['Trades'].sort_values(ascending=False).head(10))

## 3. Stock Count Analysis
Since some stocks list later than others, we calculate the number of active stock symbols present in our dataset year-by-year.

In [ ]:
active_years = {}
for symbol in SYMBOLS:
    stock_df = load_stock_data(symbol)
    if not stock_df.empty:
        years = stock_df['Date'].dt.year.unique()
        for year in years:
            active_years[year] = active_years.get(year, 0) + 1

active_counts_series = pd.Series(active_years).sort_index()

plt.figure(figsize=(12, 6))
sns.barplot(x=active_counts_series.index, y=active_counts_series.values, color='teal')
plt.title("Number of Active Stocks in nifty50 Dataset Per Year")
plt.xlabel("Year")
plt.ylabel("Active Symbol Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Sector Distribution Analysis
We visualize the representation of different industries/sectors in the NIFTY-50 constituent stocks.

In [ ]:
sector_counts = metadata_df['Industry'].value_counts()

plt.figure(figsize=(12, 6))
sns.barplot(x=sector_counts.values, y=sector_counts.index, hue=sector_counts.index, palette='crest', legend=False)
plt.title("Sectoral Weight distribution (by constituent count) in NIFTY-50")
plt.xlabel("Stock Count")
plt.ylabel("Industry Sector")
plt.tight_layout()
plt.show()

## 5. Price & Returns Distribution Analysis
We align the Close prices of the stocks and calculate daily log returns to inspect returns distributions and check skewness and kurtosis.

In [ ]:
# Load aligned panel Close prices for a subset of major tickers
target_stocks = ['INFY', 'TCS', 'RELIANCE', 'HDFCBANK', 'SBIN']
closes = create_aligned_panel(target_stocks, start_date="2005-01-01", end_date="2021-04-30")

# Calculate log returns
returns = np.log(closes / closes.shift(1))

# Plot returns distribution curves
plt.figure(figsize=(12, 6))
for stock in returns.columns:
    sns.kdeplot(returns[stock].dropna(), label=stock, fill=True, alpha=0.08)

plt.title("Daily Log Returns Distribution Curves (2005-2021)")
plt.xlabel("Daily Log Return")
plt.ylabel("Density")
plt.xlim(-0.08, 0.08)
plt.legend()
plt.show()

# Skewness and Kurtosis statistics
print("=== SKEWNESS OF DAILY RETURNS ===")
print(returns.skew())

print("\n=== KURTOSIS OF DAILY RETURNS ===")
print(returns.kurtosis())
print("Note: High positive kurtosis (fat tails) is typical for equity daily returns.")

## 6. Correlation Analysis
We align all 50 stocks from 2010 to 2021 to examine returns co-movement correlations.

In [ ]:
# Align all close prices from 2010 onwards
all_closes = create_aligned_panel(SYMBOLS, start_date="2010-01-01", end_date="2021-04-30")
all_returns = np.log(all_closes / all_closes.shift(1)).dropna(how='all')

# Calculate Pearson correlation matrix
corr_matrix = all_returns.corr()

plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, cmap='coolwarm', xticklabels=False, yticklabels=False)
plt.title("Returns Correlation Matrix of NIFTY-50 Constituent Stocks (2010-2021)")
plt.show()

print(f"Average stock returns correlation coefficient: {corr_matrix.mean().mean():.4f}")

## 7. Market Regime Analysis
Finally, we analyze market regimes by comparing the NIFTY 50 index level with the India VIX volatility index.

In [ ]:
# Align benchmarks
nifty_prices = nifty50_df[['Date', 'Close']].rename(columns={'Close': 'NIFTY'})
vix_levels = vix_df[['Date', 'Close']].rename(columns={'Close': 'VIX'})

regime_df = pd.merge(nifty_prices, vix_levels, on='Date').sort_values('Date')
regime_df['NIFTY_Ret'] = regime_df['NIFTY'].pct_change()

# Plot NIFTY 50 and VIX on dual Y-axes
fig, ax1 = plt.subplots(figsize=(14, 7))
ax2 = ax1.twinx()

ax1.plot(regime_df['Date'], regime_df['NIFTY'], color='darkgreen', label='NIFTY 50', linewidth=1.5)
ax2.plot(regime_df['Date'], regime_df['VIX'], color='crimson', label='INDIA VIX', alpha=0.5, linewidth=1.0)

ax1.set_xlabel('Date')
ax1.set_ylabel('NIFTY 50 Price Index', color='darkgreen')
ax2.set_ylabel('INDIA VIX Value', color='crimson')
plt.title("NIFTY-50 Equity Index vs INDIA VIX Volatility (Regime Chart)")
plt.show()

# Scatter plot showing VIX level and next-day/same-day return relationship
plt.figure(figsize=(10, 6))
sns.scatterplot(data=regime_df, x='VIX', y='NIFTY_Ret', alpha=0.4, color='indigo')
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.title("INDIA VIX Level vs NIFTY-50 Daily Returns")
plt.xlabel("INDIA VIX")
plt.ylabel("NIFTY-50 Daily Return")
plt.tight_layout()
plt.show()

correlation = regime_df['VIX'].corr(regime_df['NIFTY_Ret'])
print(f"Correlation between INDIA VIX level and NIFTY-50 returns: {correlation:.4f}")